# Colab 閱讀說明

這份 Notebook 是由 `分析方法提案_客群分群結合生存分析與回購策略.md` 轉寫而成，主要用途是讓助教與組員在 Google Colab 中閱讀與討論分析方法。

此文件是「方法確認用」，不是最終模型結果版。若需要查看可重跑的模型計算流程，請另見 `TBD_分群成果_客群分群結合生存分析與回購策略.ipynb`。


# 分析方法提案：客群分群結合生存分析與回購策略

文件目的：明天與助教、組員討論分析方法是否可行  
資料來源：`2023-2025_旅平險_backup.csv`  
目前狀態：方法確認用，不是最終分析結果版


## 1. 文件目的

這份文件的目的，是先把我們想採用的分析架構整理清楚，明天可以拿來跟助教確認：

1. 這樣的分析方法是否合理。
2. 特徵設計有沒有統計或商務解釋上的問題。
3. 分群後再做生存分析是否需要特別注意。
4. 有沒有更適合的模型或修正方向。

我們目前想採用的主軸是：

> 先用客戶完整投保行為進行客群分群，再針對不同客群分析回購速度，最後設計差異化提醒策略。

這份文件不是要直接產出正式結論，而是作為方法確認與討論用。


## 2. 研究問題與分析目標

原本的回購分析只問：

> 客戶是否會在 180 天內再次投保？

但這個問法比較粗，因為旅平險回購不只是「會不會再買」，更重要的是：

> 哪些客戶會回購？多久後回購？應該什麼時候提醒？

因此，本研究希望回答三個問題：

1. 旅平險客戶是否能依回購行為、旅遊型態、保費價值與購買習慣分成不同客群？
2. 不同客群的回購速度與回購週期是否不同？
3. 各客群是否適合設計不同的再行銷提醒策略？

用商務語言來說，本分析想從資料中找出：

| 問題 | 商務意義 |
|---|---|
| 誰是高頻回購客？ | 可作為穩定經營與會員維繫對象 |
| 誰是長週期高保費客？ | 不宜頻繁提醒，但可在旅遊旺季或長假前觸及 |
| 誰是短週期短天期客？ | 可在較短時間內安排提醒 |
| 不同客群何時回購？ | 決定提醒時機，而不是所有人都用同一時間點 |


## 3. 為什麼採用「版本 A：描述型分群」

本研究決定採用「描述型分群」，而不是「預測型分群」。

這兩種方法的差異如下：

| 分析類型 | 可使用的特徵 | 適合回答的問題 | 本研究是否採用 |
|---|---|---|---|
| 描述型分群 | 可使用 2023-2025 完整期間的行為資料，例如投保次數、回購間隔、平均保費、目的地多樣性 | 這些客戶實際上可分成哪些樣態？ | 採用 |
| 預測型分群 | 只能使用第一次投保當下就知道的資料，例如第一次目的地、第一次保費、第一次天數 | 新客第一次投保時，能否預測他未來屬於哪一群？ | 暫不採用 |

選擇描述型分群的原因：

1. 本期末報告主要目標是發現客戶樣態與商務洞察，不是建立可上線部署的新客即時預測模型。
2. 描述型分群能納入完整行為，例如回購週期、投保頻率、目的地多樣性，比較容易找出清楚客群。
3. 結果較容易轉成商務語言，例如「高頻短週期旅客」、「長程高保費低頻型」。
4. 後續可以用生存分析把各客群轉換成提醒時機與回購策略。

重要限制：

> 因為描述型分群會使用完整期間的回購行為，所以不能宣稱這是一個「客戶第一次投保當下就能預測未來類型」的模型。  
> 它的定位是「事後行為分群」，用來理解客戶樣態與設計策略。


## 4. 整體分析流程

預計分析流程如下：

1. 建立客戶層級資料。
2. 將一次性客戶先獨立成一群。
3. 對重複投保者建立分群特徵。
4. 使用 K-means 進行客戶分群。
5. 解讀每群特徵並命名。
6. 對每群做生存分析。
7. 對每群設計兩階段回購策略。
8. 彙整成期末報告可使用的 insight 與商務建議。

可以用以下方式理解：

```text
原始保單資料
    ↓
整理成客戶層級資料
    ↓
一次性客戶獨立處理
    ↓
重複投保者 K-means 分群
    ↓
客群輪廓命名
    ↓
各客群生存分析
    ↓
各客群提醒時機與回購策略
```


## 5. 資料層級與分析單位

原始資料是一列一張保單，但分群分析需要轉成一列一位客戶。

| 項目 | 口徑 |
|---|---|
| 客戶識別 | 使用 `APL_ID` |
| 保單日期 | 使用 `LOGIN_DATE` 作為投保日期 |
| 回購間隔 | 同一 `APL_ID` 相鄰兩次 `LOGIN_DATE` 的天數差 |
| 目的地 | 使用 `SITE_CODE`，必要時轉成目的地類型 |
| 保費 | 使用 `TOT_PREM` |
| 保額 | 使用 `TOT_SA` |
| 保單天數 | 使用 `DAYS` |

注意事項：

1. `NO` 是遮罩後的保單號，不適合作為唯一保單鍵。
2. `APL_ID` 是被保人 ID，但仍可能存在同一人替家人投保或特殊代買情境，這點需要在報告中說明。
3. 年度趨勢與回購分析皆以 `LOGIN_DATE` 為主，因為我們關心的是線上完成投保行為。


## 6. 一次性客戶如何處理

一次性客戶是指在 2023-2025 期間只出現一次投保紀錄的 `APL_ID`。

這類客戶有一個重要問題：

> 他們沒有第二次投保，因此無法計算回購間隔。

如果把一次性客戶的回購間隔硬設為 0，模型會誤以為他們是「超短週期回購」。  
如果把回購間隔硬設為非常大的數字，模型又會誤以為他們是「超長週期回購」。  
這兩種處理都不合理。

因此建議：

1. 一次性客戶先獨立成「一次性/尚未回購客群」。
2. K-means 只針對有重複投保者執行。
3. 最後報告時，再把一次性客群與 K-means 產生的重複投保客群一起比較。

這樣做的好處：

1. 避免缺少回購間隔造成模型扭曲。
2. 保留一次性客戶作為重要客群，而不是丟掉。
3. 分群結果比較容易解釋。


## 7. 分群特徵設計

分群特徵會整理成五大類：回購行為、價值行為、旅遊行為、購買行為、人口特徵。

### 7.1 回購行為特徵

| 特徵 | 說明 | 商務意義 |
|---|---|---|
| 投保次數 | 同一 `APL_ID` 的保單數 | 客戶活躍程度 |
| 回購事件數 | 投保次數 - 1 | 實際重複投保次數 |
| 平均回購間隔 | 所有相鄰投保間隔平均 | 回購節奏快慢 |
| 回購間隔中位數 | 所有相鄰投保間隔中位數 | 較不受極端值影響的回購節奏 |
| 最近一次回購間隔 | 最近兩次投保的間隔 | 最近行為是否變快或變慢 |
| 30 天內回購占比 | 相鄰投保間隔 <= 30 天的比例 | 是否屬於短週期高頻客 |
| 60 天內回購占比 | 相鄰投保間隔 <= 60 天的比例 | 短中期回購傾向 |
| 120 天內回購占比 | 相鄰投保間隔 <= 120 天的比例 | 一季內回購傾向 |

### 7.2 價值行為特徵

| 特徵 | 說明 | 商務意義 |
|---|---|---|
| 平均保費 | 客戶所有保單平均 `TOT_PREM` | 單次保單價值 |
| 總保費 | 客戶所有保費加總 | 累積貢獻 |
| 平均保額 | 客戶所有保單平均 `TOT_SA` | 保障需求高低 |
| 高保費保單占比 | 高於整體 75 百分位保費的保單比例 | 是否常購買高價方案 |

### 7.3 旅遊行為特徵

| 特徵 | 說明 | 商務意義 |
|---|---|---|
| 平均保單天數 | 客戶所有保單平均 `DAYS` | 旅程長短 |
| 長天期保單占比 | 例如 `DAYS > 14` 的比例 | 是否常有長途或長假需求 |
| 目的地數 | 出現過幾種不同 `SITE_CODE` | 目的地多樣性 |
| 日本占比 | 日本保單占比 | 是否為日本核心客 |
| 國內占比 | 國內/台灣保單占比 | 是否偏國內短天期 |
| 東南亞占比 | 東南亞目的地保單占比 | 是否偏東南亞旅遊 |
| 歐美澳長程占比 | 歐美澳長程目的地保單占比 | 是否偏長程旅遊 |
| 多目的地占比 | 多目的地串接代碼占比 | 是否有複雜行程 |

### 7.4 購買行為特徵

| 特徵 | 說明 | 商務意義 |
|---|---|---|
| 平均提前投保天數 | 平均 `EFF_DATE - LOGIN_DATE` | 客戶是否習慣提早規劃 |
| 當天投保占比 | 提前投保天數 = 0 的比例 | 是否常臨時購買 |
| 3 天內投保占比 | 提前投保天數 <= 3 的比例 | 是否接近出發才購買 |
| 非信用卡/特殊付款占比 | `CARD_BANK = 999999` 比例 | 付款行為差異 |

### 7.5 人口特徵

| 特徵 | 說明 | 商務意義 |
|---|---|---|
| 首次投保年齡 | 第一次投保當年年齡 | 客戶生命週期 |
| 平均投保年齡 | 所有投保紀錄平均年齡 | 近似客戶年齡輪廓 |
| 年齡層 | 18-29、30-39、40-49 等 | 方便報告解釋 |

### 7.6 類別資料如何處理

K-means 比較適合使用數值特徵。  
因此，不會直接把「日本」、「國內」、「信用卡銀行名稱」這類文字放進模型。

處理方式是轉成比例特徵，例如：

| 原始類別 | 轉換方式 |
|---|---|
| 目的地 | 日本占比、國內占比、東南亞占比、長程占比 |
| 付款方式 | 非信用卡占比、特定銀行群組占比 |
| 年齡層 | 可先用數值年齡，分群後再回頭看年齡層分布 |


## 8. 模型選擇：K-means + 旅平險版 RFM

### 8.1 為什麼選 K-means

K-means 是一種常見的分群方法。  
白話來說，它會把「行為特徵相近」的客戶分在同一群。

它的基本概念是：

1. 先指定要分成 K 群。
2. 模型會找到 K 個群中心。
3. 每位客戶會被分到距離最近的群中心。
4. 模型反覆調整，讓同群內的客戶盡量相似，不同群之間盡量不同。

選擇 K-means 的原因：

1. 容易解釋，適合期末報告。
2. 可以搭配表格說明各群特徵。
3. 分群後可依商務意義命名。
4. 對本研究這種客戶行為分群問題相對直覺。

### 8.2 為什麼搭配旅平險版 RFM

傳統 RFM 常用於客戶價值分析：

| RFM | 原本意義 | 旅平險版本 |
|---|---|---|
| Recency | 最近一次購買距今多久 | 最近一次投保距資料截止日多久 |
| Frequency | 購買頻率 | 投保次數、回購事件數 |
| Monetary | 消費金額 | 總保費、平均保費 |

本研究會再加入旅平險特有特徵：

| 延伸面向 | 旅平險特徵 |
|---|---|
| Interval | 平均回購間隔、回購間隔中位數 |
| Destination | 目的地數、日本占比、國內占比、長程占比 |
| Trip | 平均保單天數、長天期占比 |
| Purchase Timing | 平均提前投保天數、當天投保占比 |

因此本研究可稱為：

> K-means + 旅平險版 RFM 行為特徵分群


## 9. K 值如何選擇

K 值就是要把重複投保者分成幾群。

建議測試 K=3 到 K=6。

評估方式包含三種：

### 9.1 肘部法

肘部法會觀察分群數增加後，群內差異下降多少。

白話來說：

> 如果從 K=3 增加到 K=4，群內差異下降很多，代表多分一群有幫助。  
> 但如果從 K=5 增加到 K=6，只改善一點點，就不一定值得。

### 9.2 輪廓係數

輪廓係數用來看分群分得清不清楚。

它大致可以理解為：

> 同一群的人是否夠像？不同群的人是否夠不一樣？

數值越高通常代表分群越清楚。

### 9.3 商務可解釋性

不一定是統計分數最高的 K 就最好。  
最後仍要看分出來的群是否能被清楚解讀。

例如：

| K 值 | 可能問題 |
|---|---|
| K=3 | 太粗，可能把不同客群混在一起 |
| K=4 或 K=5 | 通常較容易平衡解釋與細節 |
| K=6 | 可能更細，但報告上較難說清楚 |

建議最後選擇：

> 統計指標合理，且每群都能用商務語言命名的 K 值。


## 10. 分群後的客群命名方式

K-means 不會自動產生有意義的客群名稱。  
模型只會告訴我們「哪些客戶被分在同一群」。

客群名稱需要根據每群的特徵命名。

可能命名範例：

| 可能客群名稱 | 判斷依據 |
|---|---|
| 高頻短週期旅客 | 投保次數高、平均回購間隔短 |
| 日本穩定回購型 | 日本占比高、回購週期穩定 |
| 國內短天期快回購型 | 國內占比高、平均天數短、回購速度快 |
| 長程高保費低頻型 | 長程占比高、平均保費高、回購間隔長 |
| 多目的地活躍型 | 目的地數多、投保次數高 |
| 一次性/尚未回購客群 | 只有一筆投保紀錄，無法計算回購間隔 |

命名原則：

1. 名稱要反映該群最突出的特徵。
2. 不要只用模型群號，例如 Cluster 1、Cluster 2。
3. 名稱要能讓非技術讀者理解。


## 11. 分群後的生存分析

分群完成後，會對每個客群做生存分析。

這裡的生存分析不是要證明「分群造成回購差異」，而是要把分群結果轉換成：

> 每一群大概多久後會再次投保？

會計算以下指標：

| 指標 | 說明 | 商務用途 |
|---|---|---|
| 30 天累積回購率 | 第一次投保後 30 天內有多少比例回購 | 超短期提醒 |
| 60 天累積回購率 | 60 天內回購比例 | 短期提醒 |
| 120 天累積回購率 | 120 天內回購比例 | 一季內追蹤 |
| 180 天累積回購率 | 180 天內回購比例 | 與原本模型比較 |
| 365 天累積回購率 | 一年內回購比例 | 年度回購潛力 |
| KM 回購中位時間 | 累積回購率接近 50% 的時間 | 主力提醒時間參考 |

### 11.1 為什麼要用生存分析

有些客戶沒有第二次投保，不代表永遠不會回購。  
可能只是資料截止前還沒回購。

這種情況叫做「右設限」。

白話說：

> 我們只知道他在資料結束前還沒回來，但不知道資料結束後會不會回來。

生存分析可以比較合理地處理這種資料截止問題。


## 12. 兩階段回購策略

分群與生存分析完成後，可以轉成兩階段商務策略。

### 12.1 第一階段：是否值得追蹤

不是每一群都適合用同樣密集的提醒。

判斷追蹤優先級時，可參考：

| 指標 | 解讀 |
|---|---|
| 客群人數 | 群體規模是否夠大 |
| 平均保費 | 單次價值是否高 |
| 總保費 | 累積貢獻是否高 |
| 365 天累積回購率 | 一年內回購潛力 |
| 回購中位時間 | 回購速度快慢 |
| 目的地偏好 | 是否能搭配旅遊旺季或目的地行銷 |

可能策略：

| 客群類型 | 追蹤策略 |
|---|---|
| 高頻短週期旅客 | 高優先級，可較早提醒 |
| 日本穩定回購型 | 高優先級，可搭配日本旅遊旺季 |
| 國內短天期快回購型 | 高優先級，適合短週期提醒 |
| 長程高保費低頻型 | 中高優先級，但提醒時間應拉長 |
| 一次性/尚未回購客群 | 需先判斷是否有再開發價值 |

### 12.2 第二階段：什麼時候提醒

提醒時機可依每群的回購分布設計。

建議參考：

| 指標 | 提醒用途 |
|---|---|
| 回購間隔 25 百分位 | 第一波較早提醒 |
| 回購間隔中位數 | 主力提醒 |
| 回購間隔 75 百分位 | 後續喚回提醒 |
| KM 累積回購率快速上升區間 | 代表客群常見回購時間窗 |

範例：

| 客群 | 可能提醒節奏 |
|---|---|
| 高頻短週期旅客 | 第 30-45 天先提醒，第 60-90 天再次提醒 |
| 日本穩定回購型 | 第 45-60 天先提醒，第 110-130 天主提醒 |
| 長程高保費低頻型 | 第 90-120 天後再提醒，避免太早打擾 |
| 一次性/尚未回購客群 | 不用高頻提醒，可搭配旅遊旺季、連假或促銷活動 |


## 13. 必須注意的限制與風險

這一節是明天需要特別跟助教確認的重點。

### 13.1 描述型分群不能宣稱為即時預測

因為本方法使用完整期間的回購行為，例如投保次數、平均回購間隔。  
這些資料在客戶第一次投保當下還不知道。

因此不能說：

> 新客第一次投保時，我們就能預測他屬於哪一群。

比較正確的說法是：

> 本研究透過 2023-2025 年完整投保行為，回顧性地辨識旅平險客戶樣態。

### 13.2 分群特徵包含回購間隔，後續生存分析會有循環解釋風險

如果分群時已經使用回購間隔，再拿生存分析說「某群回購比較快」，會有循環解釋的風險。

也就是：

> 先用回購快慢把人分群，再說某群回購較快。

這不是完全不能做，但解讀時要小心。

建議定位：

> 生存分析不是用來證明分群造成回購差異，而是用來把分群結果轉換成可操作的時間策略。

### 13.3 生存分析處理的是觀察資料，不代表因果

即使某群回購比較快，也不能說分群特徵造成回購比較快。  
只能說這些特徵與回購速度有關聯。

### 13.4 右設限問題

沒有觀察到第二次投保，不代表客戶永遠不會回購。  
可能只是資料截止到 2025-12-31，後續尚未被觀察到。

生存分析可以處理右設限，但仍無法知道資料期間外的真實行為。

### 13.5 `APL_ID` 可能有代買或家庭投保情境

`APL_ID` 是被保人 ID，但資料仍可能存在：

1. 同一人替家人投保。
2. 家庭旅遊多張保單。
3. 特殊代辦或團體情境。

這會影響「客戶」解讀，需要在報告中說明。

### 13.6 `CARD_BANK` 對照銀行正確性待確認

`CARD_BANK` 疑似信用卡 BIN/IIN，但銀行名稱目前是公開第三方查詢推估，不是官方欄位定義。  
若納入模型，應以「付款代碼」或「推估銀行」表述，並註明正確性待確認。

### 13.7 K-means 對特徵尺度敏感

不同特徵單位差異很大，例如：

| 特徵 | 數值範圍 |
|---|---|
| 投保次數 | 1 到數十次以上 |
| 平均保費 | 數百到數千元 |
| 平均回購間隔 | 數天到數百天 |
| 日本占比 | 0 到 1 |

如果不標準化，數值大的特徵會主導分群。

因此 K-means 前必須做標準化，讓不同特徵站在同一尺度上比較。

### 13.8 K 值選擇有主觀性

K-means 必須先指定分成幾群。  
即使用肘部法或輪廓係數，最後仍需要研究者判斷哪個 K 值最有商務解釋力。

因此報告中要說明：

> K 值選擇會同時考量統計指標與商務可解釋性。


## 14. 預期輸出

如果助教確認這個方向可行，後續預計產出以下結果：

1. 客戶層級分群特徵表。
2. K 值選擇比較表。
3. 各客群輪廓表。
4. 各客群命名與解釋。
5. 各客群 Kaplan-Meier 回購曲線。
6. 各客群 30/60/120/180/365 天累積回購率。
7. 各客群回購間隔 25/50/75 百分位。
8. 各客群兩階段提醒策略表。
9. 期末報告可使用的主要 insight。

可能圖表：

| 圖表 | 用途 |
|---|---|
| K 值肘部圖 | 說明分群數選擇 |
| 客群雷達圖或標準化特徵圖 | 比較各群輪廓 |
| 客群人數長條圖 | 說明各群規模 |
| 生存曲線圖 | 比較各群回購速度 |
| 提醒時機表 | 轉成商務策略 |


## 15. 想請助教確認的問題

明天討論時，建議直接帶著以下問題詢問助教：

1. 本研究採用「描述型分群」，並納入回購區間、平均回購間隔等特徵，是否合理？
2. 一次性客戶先獨立處理，不放入 K-means，是否合理？
3. K-means 是否適合作為本資料的客戶分群方法？
4. 若分群特徵已包含回購間隔，分群後再做生存分析，解釋上是否需要更多限制說明？
5. 生存分析在這裡定位為「轉換成提醒策略」，而不是因果驗證，這樣是否合適？
6. 是否需要改用 RFM 分數法、階層式分群，或其他模型？
7. K 值選擇除了肘部法、輪廓係數、商務可解釋性外，是否還需要其他判斷標準？
8. 若期末報告篇幅有限，應優先保留哪些分析結果？
9. `CARD_BANK` 若正確性待確認，是否建議排除於 K-means 特徵之外，只作為分群後描述？
10. 生存分析的右設限說明是否足夠清楚？


## 16. 暫定結論與下一步

目前暫定方法是可行的，但需要明天先與助教確認。

暫定結論：

1. 本研究適合採用描述型分群，而不是新客即時預測模型。
2. 一次性客戶應獨立處理，避免回購間隔缺失造成模型扭曲。
3. 重複投保者可用 K-means 搭配旅平險版 RFM 特徵分群。
4. 分群後可用生存分析比較不同客群的回購速度。
5. 最後可用兩階段策略把結果轉成商務建議：先決定誰值得追蹤，再決定何時提醒。

若助教確認方法可行，下一步會進行：

1. 實際建立客戶層級分群資料。
2. 測試 K=3 到 K=6。
3. 選定最適合的 K 值。
4. 產出正式客群分群結果。
5. 對各客群執行生存分析。
6. 撰寫正式期末報告分析結果。
